# Coco Crepe — 06 Gold Sales Summary

Publicación del Data Product final de ventas.

In [0]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# Completa estos valores solo si la detección automática no encuentra
# exactamente un catálogo por Data Product.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

In [0]:
sales_detail = f"{SALES_CATALOG}.silver.sales_detail"
sales_summary = f"{SALES_CATALOG}.gold.sales_summary" 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {sales_summary} AS
WITH order_totals AS (
    SELECT
        order_id,
        order_date,
        customer_id,
        SUM(quantity) AS units_sold,
        ROUND(SUM(line_total), 2) AS order_revenue
    FROM {sales_detail}
    GROUP BY
        order_id,
        order_date,
        customer_id
)
SELECT
    order_date,
    COUNT(*) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(units_sold) AS units_sold,
    ROUND(SUM(order_revenue), 2) AS total_revenue,
    ROUND(AVG(order_revenue), 2) AS average_ticket,
    current_timestamp() AS published_at
FROM order_totals
GROUP BY order_date
""")

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS record_count,
    MIN(order_date) AS minimum_date,
    MAX(order_date) AS maximum_date,
    ROUND(SUM(total_revenue), 2) AS accumulated_revenue
FROM {sales_summary}
""").display()

spark.sql(
    f"SELECT * FROM {sales_summary} ORDER BY order_date LIMIT 30"
).display()